In [ ]:
# ============================================================
# ANN REGRESSION — CO2 STORAGE AND LEAKAGE
# ============================================================

# ----------------------------
# 1. IMPORT LIBRARIES
# ----------------------------
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error


# ----------------------------
# 2. CREATE OUTPUT FOLDER
# ----------------------------
os.makedirs("figures", exist_ok=True)


# ----------------------------
# 3. LOAD DATA
# ----------------------------
# Replace with your real file name
file_path = "example_dataset.xlsx"

df = pd.read_excel(file_path)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


# ----------------------------
# 4. DEFINE INPUT FEATURES
# ----------------------------
x_cols = [
    "PERM_permeability",
    "PERM_perm_factor_of_failed_well",
    "PERM_dist_to_well_in_grid",
    "PERM_caprock_perm",
    "porosity",
    "aquifer_porosity",
    "aquifer_permeability"
]

X = df[x_cols].copy()


# ============================================================
# PART A — ANN FOR TOTAL SECURE CO2 MASS
# ============================================================

# ----------------------------
# 5A. DEFINE TARGET
# ----------------------------
y_total = df["CO2 Total Secure Mass"].values


# ----------------------------
# 6A. TRAIN / TEST SPLIT
# ----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_total,
    test_size=0.20,
    random_state=42,
    shuffle=True
)


# ----------------------------
# 7A. SCALE INPUTS
# ----------------------------
scaler_X_total = StandardScaler()
X_train_s = scaler_X_total.fit_transform(X_train)
X_test_s = scaler_X_total.transform(X_test)


# ----------------------------
# 8A. TRAIN ANN MODEL
# ----------------------------
mlp_total = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=0.001,
    max_iter=2000,
    random_state=42
)

mlp_total.fit(X_train_s, y_train)


# ----------------------------
# 9A. PREDICT
# ----------------------------
y_train_pred = mlp_total.predict(X_train_s)
y_test_pred = mlp_total.predict(X_test_s)


# ----------------------------
# 10A. METRICS
# ----------------------------
r2_total_train = r2_score(y_train, y_train_pred)
r2_total_test = r2_score(y_test, y_test_pred)
mae_total_test = mean_absolute_error(y_test, y_test_pred)

print("\n=== ANN — TOTAL SECURE CO2 MASS ===")
print(f"Train R²: {r2_total_train:.4f}")
print(f"Test  R²: {r2_total_test:.4f}")
print(f"Test  MAE: {mae_total_test:.4f}")


# ----------------------------
# 11A. PARITY PLOT
# ----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(
    y_train, y_train_pred,
    color="#4A76FF",
    edgecolor="black",
    linewidth=0.4,
    s=28,
    alpha=0.7,
    label="Train"
)

plt.scatter(
    y_test, y_test_pred,
    color="#FF6A6A",
    edgecolor="black",
    linewidth=0.4,
    s=30,
    alpha=0.8,
    label="Test"
)

lo = min(y_train.min(), y_test.min(), y_train_pred.min(), y_test_pred.min())
hi = max(y_train.max(), y_test.max(), y_train_pred.max(), y_test_pred.max())

plt.plot([lo, hi], [lo, hi], "k--", linewidth=1.1)

plt.xlabel("True Total Secure CO₂ Mass (t)")
plt.ylabel("Predicted Total Secure CO₂ Mass (t)")
plt.title("Parity Plot — Total Secure CO₂ Mass")

plt.text(
    0.05, 0.92,
    f"$R^2 = {r2_total_test:.3f}$",
    transform=plt.gca().transAxes,
    fontsize=14,
    weight="bold"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "figures/Figure_ANN_Parity_TotalSecureCO2.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()


# ============================================================
# PART B — ANN FOR TOTAL LEAKAGE CO2 MASS
# ============================================================

# ----------------------------
# 5B. DEFINE TARGET
# ----------------------------
y_leakage = df["CO2 Total Leakage Mass"].values


# ----------------------------
# 6B. TRAIN / TEST SPLIT
# ----------------------------
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X,
    y_leakage,
    test_size=0.20,
    random_state=42,
    shuffle=True
)


# ----------------------------
# 7B. SCALE INPUTS
# ----------------------------
scaler_X_leak = StandardScaler()
X_train2_s = scaler_X_leak.fit_transform(X_train2)
X_test2_s = scaler_X_leak.transform(X_test2)


# ----------------------------
# 8B. SCALE TARGET
# ----------------------------
scaler_y_leak = StandardScaler()
y_train2_s = scaler_y_leak.fit_transform(y_train2.reshape(-1, 1)).ravel()


# ----------------------------
# 9B. TRAIN ANN MODEL
# ----------------------------
mlp_leak = MLPRegressor(
    hidden_layer_sizes=(128, 64, 32),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=0.001,
    max_iter=2000,
    random_state=42
)

mlp_leak.fit(X_train2_s, y_train2_s)


# ----------------------------
# 10B. PREDICT
# ----------------------------
y_train2_pred_s = mlp_leak.predict(X_train2_s)
y_test2_pred_s = mlp_leak.predict(X_test2_s)

y_train2_pred = scaler_y_leak.inverse_transform(y_train2_pred_s.reshape(-1, 1)).ravel()
y_test2_pred = scaler_y_leak.inverse_transform(y_test2_pred_s.reshape(-1, 1)).ravel()


# ----------------------------
# 11B. METRICS
# ----------------------------
r2_leak_train = r2_score(y_train2, y_train2_pred)
r2_leak_test = r2_score(y_test2, y_test2_pred)
mae_leak_test = mean_absolute_error(y_test2, y_test2_pred)

print("\n=== ANN — TOTAL LEAKAGE CO2 MASS ===")
print(f"Train R²: {r2_leak_train:.4f}")
print(f"Test  R²: {r2_leak_test:.4f}")
print(f"Test  MAE: {mae_leak_test:.4f}")


# ----------------------------
# 12B. PARITY PLOT
# ----------------------------
plt.figure(figsize=(6, 6))

plt.scatter(
    y_train2, y_train2_pred,
    color="#4A76FF",
    edgecolor="black",
    linewidth=0.4,
    s=28,
    alpha=0.7,
    label="Train"
)

plt.scatter(
    y_test2, y_test2_pred,
    color="#FF6A6A",
    edgecolor="black",
    linewidth=0.4,
    s=30,
    alpha=0.8,
    label="Test"
)

lo2 = min(y_train2.min(), y_test2.min(), y_train2_pred.min(), y_test2_pred.min())
hi2 = max(y_train2.max(), y_test2.max(), y_train2_pred.max(), y_test2_pred.max())

plt.plot([lo2, hi2], [lo2, hi2], "k--", linewidth=1.1)

plt.xlabel("True Total Leakage CO₂ Mass (t)")
plt.ylabel("Predicted Total Leakage CO₂ Mass (t)")
plt.title("Parity Plot — Total Leakage CO₂ Mass")

plt.text(
    0.05, 0.92,
    f"$R^2 = {r2_leak_test:.3f}$",
    transform=plt.gca().transAxes,
    fontsize=14,
    weight="bold"
)

plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()

plt.savefig(
    "figures/Figure_ANN_Parity_TotalLeakageCO2.png",
    dpi=600,
    bbox_inches="tight"
)

plt.show()
plt.close()